# AP Model 3D Foto MAX: usa toda la GPU y la RAM (A100 / L4 / T4)

Foto → modelo 3D con textura HD, **esqueleto automático compatible con Mixamo**,
**pesos sin huecos**, **6 animaciones**, piezas por hueso y exportación a
**Blender (.blend) / FBX / GLB animado / OBJ / STL / video MP4**.

**Qué hace distinto esta versión**
* Se ajusta sola al equipo. Con una **A100 de 80 GB + 167 GB de RAM** usa el perfil **MÁXIMO**: forma octree 512, hasta 150 000 caras (antes ~25 000), textura de 4096 px y video de 1080p.
* **Usa la RAM a fondo**: carga en la RAM los pesos de los modelos (cerca de 30 GB) y deja **todos los modelos cargados a la vez** en la GPU (1 foto, varias vistas y textura). Trabaja en `/dev/shm`, un disco en RAM, guarda en caché los resultados y deja que Blender corra **en paralelo** con la siguiente generación.
* **Pesos sin manchas**: los calcula Blender (método de calor) sobre una copia cerrada de la malla y los pasa al modelo real. Cada hueso mueve su zona completa, no solo una banda.
* **Malla reparada**: quita trozos sueltos, cierra agujeros y orienta las caras.
* **Cero vértices sin peso**: si alguno queda suelto, copia el peso de su vecino; el informe dice cuántos vértices mueve cada hueso.
* **LOD para juegos**: además del modelo completo, versiones al 50 % y 15 % con el mismo esqueleto y animaciones.
* **Visor web** (`visor.html` en el ZIP): botones para cada animación, cambio de calidad y modo AR en el celular.
* **Multivista**: si subes espalda o perfiles (o una hoja de vistas), usa Hunyuan3D-2mv y pinta también la espalda.

## Cómo usarlo
1. *Entorno de ejecución › Cambiar tipo de entorno de ejecución* → **GPU (A100 recomendado)** y **RAM alta**.
2. Ejecuta las celdas **1, 2 y 3** en orden. La primera vez tardan unos 6–10 min: la 2 compila Hunyuan3D-2 y la 3 descarga los modelos y los carga en la RAM.
3. Abre la **4. Estudio** dentro del notebook, o la **5. Página web** con contraseña para usarla desde el celular.
4. Para no perder tus modelos, usa la **6. Guardar en Drive**.

> **Aviso "No se ha podido completar el guardado automático"**: aparece cuando el notebook está abierto en dos pestañas o dispositivos a la vez. Ciérralo en los demás y usa *Archivo › Guardar una copia en Drive*. No afecta a lo que se está ejecutando.

## 1. Proyecto (código incluido: no necesita Drive ni subir nada)

In [ ]:
# @title 1. Proyecto (código incluido: no necesita Drive ni subir nada)
import base64, json, os, subprocess, sys, zlib
APP = "/content/ap3d_app"
os.makedirs(f"{APP}/ap3d", exist_ok=True)
_FILES = json.loads(r"""{"__init__.py": "eNoDAAAAAAE=", "config.py": "eNq9Vl9v2zYQf9enIJSHyq0gy7LieB40IP/WBUiWokuAAUEgMBJtq5FEjqTcdEM/1973yXZHUrHcJun6MgG2KPL4u/vdHY/n+/4J06zQlNSUvH13TT7h4P3hBQxYXa0Y/BPB5LKqSUMVobXm5I+OkYLescjzjnlLupaSw0kck5KReUzeHsHeyewABzCDWIqRTtEB1MXh72cXlwtP85IrUsOv4SXDN4gCfElb0CBXFJdZ2xtnrNywP0OzRTDFlQcbhGRGuHWiqDIoaLFmaICgq6oF09+Qcck2Y7VuRpZlQ+uaAhT1NHvQnaSGM9lUJeNE0RrAjEIJeuquqICqcwGNPN/3vaoRXGoCRriRWne6qr3+Uyj7+f7y8opkIBexdlNJ3kYrpgP/8N30JMc1PyT+uOCtZq0eUzEt/ZF3eX2Vn5y9f24bLH+1Kwebq5Iq2H10fvrryemzu93yDsId8C2Z7N8A4nklW5J8JbpgtPAIPFp+sgN8HEnNZbHeTi7tRFR0JY0qldMNrWoKoD1G/wgwbiAKxuUQnqpguZAc0kRXTAXxaGePZBCmloiopQ0L4a25pnXesIbLT2RMktevp7HZwR4KJjQ5NS+I3Fa3oEp5AzD/+N01OCKOYkdY0iYvK3UfNFWbLyVj+eoumzvrIezHVAoGBwZSS0t6Rz9wTDuTc9sEUxVZU0gnJSgkzo/43fKQAG7BI8ydr9yJmsAlNoUiNCCHM7Nigd+j+qPICA15Op+b+Z8yMjB519slQD8C2Wxxxvs7cpAuDb1nZSVVUIbgxkrpnN9nV7JjT8YChFDhf3V6aVNSUL2OPvCqDTD/IQ17Y6yOb1ux1b4NXGkqWZ9nNkU2EE1QaZPY7jQT9mhGcDB095hBwcgm1I6HC9EpazSM8oJ3LaggXJLELH/k8j6EqDW5C+Bj+sABctExRkB0DuKFdceerZe2WI7JL/DhDc9FWQEPWyoz39ZKSFFeaFCR7U8SUKiZUNl+HJJi3bX3KktjfMKdENm8ogVT2WTfLBOodCD6wywED+Ixz5I4nYdQ0+qct+gl4+MnYPhyWXNaZj/TWqFfsUjmEpHjOeAq2ogavmZpaFzCpMqm1uFwjWxdMJ0vBg5I0QHPMz88vxrwTtHQr3gn3+Cd/D+8EaznnTzBO4kd7/OUJOmLrC9OT84GtKfztKedbmnPXmQ935K2PJ8jbVl9B+uDZEB6mjxBWrHFF2B75Colk/0XSR8d/nZ2PGCd7M961tMt6/lLpNNnSE/iJP0u0iYTnuM8mW05T9wR3yPYBjVdsaZ9u7PqqDRNDBcV9AxwVwybHLgv4NaBqgg9R8sdxobXGyax5WBMohTI2gsD+hLscphsKriqTRPi6qVBPrI3toMBAcStQREJ3JLpvlDjG7QvBJ1gWCMADd59dzWKDIK48bGEmfYpNwYr/9bWNUzkWdxLQdeF7so/smq11jtCadyXPjcBx3BwGdz4zoNmT0MfguFUSFzhEFEnSqpZgEHblnO8jiWU4TLAr5BMRiHZme5ndyOMdTzDPxs/qNEy2y3eW4R+BmF27hvxeNWoQlZ3LBDutnHrwdLHPvUvcfMKjH51+5kEOHZm4/fboxH552+TJrgyXMD53dt46aMMGo0SGEAQcS00rtghrsG8PTlm3g7d/JeQa6qge0Exc25e3S7Cz9hsQ1aBdN8K4zqMEUM8PAUz6H1QtvcpbIBL/F8appwq", "engine.py": "eNqtW+ty3LaS/j9PgWXKdch4RI9kn+Qc7WGqlFh2VCvbWsdJdqNVsTAkOAOLtxDkSCOXn2v/75NtdwMkQQ5nbCdxJRqSABqNRvfXF4KO47wq6qI6ZQn8uMpjR9+xjKcpZ0+fM/fHJt82PH/6/OiEPWEn2Ybaa3FfNxXHy+9Tkcei8mezn1XBYsEqETWVKtTpjLGv2bsiLhRL4f+siAX+/t6ImOcs4tWKY5vIGcz18upnxvFiIx6YW4oqkSl7dfZfF6/esCdAibGzy3dvvDnjSrK8YGu+ZUJBP44U6kqwJue0BLZFMkquGgkNwic2LmGiUgBXTAlWAos4e27mfnv2irkRj9YC+S/5SuZAFS4vZd7ceywVW1hjQVzgCngVreVGs17yiqewrlNmiFbAiFkrq3kVAyti1cBwpTk5T1ld8SV/D80iKwsYzjYcST2JxeaJWmfMjaWKCnwEnHlmAVpExH1Ri5VgUZHjVBHPY5Djv7cbwdwfoN9jGgp9KpCMyIn1llUaaYuIrUQOgoxkkfe70cutEqpJa9ormH7V4KKoI0pci02zymA1sDFqjbJL293oxfa+UbVQKKlS1LLCLplUmekoFJO5qnkO/4nCnzmOM5MooBoZBp2qgFc/aUDxoCtsUJS07auovcLZU7lsb2XRXr1XRd5eF6q9Uuumlml31yxBuJFQXXu9rgSPZb7qHshMtNcPsgQlFbP2Pm+ycouM5eUsqYqMXV1cMtN2kfGVmOufN6Wa6Q4+6xeYyNVs9uP523MWAIN+yeu1H8sq55lw23u+VPjrhiFOHIaeN4OdCi/f/PAfMKpj1r8solvXm/1ycf7rT9DgOjBbXjtz5ix5dIu/qUjovpKrde14s9ksSrlS7DwH5RentGexSFgYylzWYegqkSZz1L0keF3ksJS0WAVlJfPa073xH3byS5gR+zFQBr0uPxa1iGrgaNARKEBX+Dt8XIlsiQ04zbBFrXkp9reE2WayseTA5WSLVt6Affg4ZoHHW3j+gqdqNGSprSwsiyKFHlHivyOxX8H9+T1AH0Cpm/H78K6obkWlAs3CtWPunZteCrCtGb8VsMvKNZJ68/O78PnF2zkT91LVYXEbvKsaAfuD/b9iR3/in0bcbmcBBNOCx+GdQBVQZoPBNAsVuE4t8gjM7YkF/6guE4+zjeNZKgBWeykEqzvY78ASoZUQGHEI7t7zKu3gH9DDR4NvyZB5rJsVaOMq4ZEI182yNRYFkBfS1oUgun4ImISCLbm+6R7V1bbnjPqATlYAMyMarufTwoed8Z9MWEVtoYxx3J5uHW2xoV4wZCMV4Omevj2/j4FhVVdu4pNJo3l7RCrR023ouerXJO4jUdbsnH4QsdEBDmdprct1XhfkMZq4ACcmCHE7P/fji1PYUuENhlYC4DVni1n3FHWlil1ibDhNDsJeDJ7cyXrNilLk1B3hZQlTLJskEYAUq2DhIbfJrkzu1rBIhqo+La8lTKXN0v3mGfvXv9jJwpvsCPuVFzVb7he7WWE+2SHH/QALd5deLwHEDsR9H/9YIEarnQaA429opeJ+yEdd1BxRQzWZK+7B+Eu3iudaE7xddHQT50pbjIktTLgExnLKPmhiEJN9/fXTxal/nHxkL7/HZmd3aYnzwVoBO2L1qb9IPiqnn9TIhajOOpRAiDDQcAeeGSCWAMncEbJqhOoX2nrKAgy/lyHe+eh9AD6VHzUxh+XXWZP6EGoWd2GdPD0BySCtA4Py/Eu7LwGx1hmvbse9NcJsn8YQ/BifYzj/HiisqgJitrciKzaimh5DPgcu2mEdJj6X714Aj694DciXr65kKVJwqp9WKFBeywNKRR5rwrRbD7nD6AQt7TP30yKY+IGiYQAJO9p3AYUyPmfHFJ55vu873u741id/avU+yi4ERYbgF25jd0dJ9/gcCMqSIgWXGzhr0xAfxbI+2pwcLZz5DplGiVDxBCKOXBXgf7X/tMRi9JjBgkfBwx+QEoxq5bThlQSbB9yvuZoQ144zmope/rwY94kS/PRhYWKHSWqHJfqZPmkoSoaZJKZxkOmURS4hqpqjp2oUxgaqgCRFq93IRbXbp0O6bgf17Zdv3xWNc002+7l7NkABGDuFAUS5s/xJQbRh6fSYnY2eNA9vmnSoRB3qVcHWyQfhtlEoPLQj0DFaQI8iSRD0nZtpF9qz7oucw76FlOuGUdmEZqj7p5XjnSkvXJwNdYTBOoAeippTFhuBplRTDq+zBMhEtpB7gOu0klJvIvI5nC6YuKIVEjihsMjDVdk4N9NAgr13tfErSuLBoiN+ykTa1gnADdaSgMPUNPqoeLYXLfy6cB0Qu+NNJy8Db2eFFK9MHSaF+XQ88cnYoB0dZiJzvT46mNYzsGS4tCOC5JNmqqOPA4H7rs1BdoxxJZiEJD2sKDETlZ8JtTZ3rV2+gkdv6clsjzKb/MteDAZq8POJEXqiboBREP3Uudk32LAXWJy5IFEOigCsAwA2aCrBmNx8v6bv/rPXEtCW7HKDuzde5x/NMCKeLSWvUK9rnvGcLM5gqzE4K67UyoSELE3Z2Xezf6WiOs1EyzDMxH+bDAsQNMDfyKpueIpTFdV2BEwYQumYUYej1CfE+DLiNSCu1wbXqMBWR6lCvuEyRa2DTiIFASymxZM4WBYDrxZz9mGT+eBK40HI/gSfjiL5hY7k/+9/9wFb4iCKfFgZCkZJ/rapeBauln+7weGW9Y73cyjhkiv1l5QXJNa2cqHsCkNpAEFmK6wtYJAaLlc7GUOCHQ4Bww4YQ3JNtcJIuEQbUmhvrDm4v1Rw8yklhQdWING3vimVL+5lEoKnzcHTKDHsuuaw32m55jAAGnzEbBYEzHn78vszh9A/L31eVXxLA68hjJizpzd+JnNQD8hV/76wWe/EoIcWdT/D1Ar6iB+pI+pAoF+7OLvjjRc07nFmqcFXWKFGz1kx0LZi+V5QrRxjigqtVleTGx5X6CzRu0LatBI9KKMAJlfa9diqObtXulte5A+iKlzOvmPHi0EIhwn2vRrt1/0Cxh7P2RZ+t8dA415pCc7pit/j1bZ7tjXPJiRGQqiK0nWR5pbossfsmOjCryW1uzlbmxEDCMQ1AExjIc/FPh77mh37x/+wXOLvnXbl4s4Iew4BCugisOfCrOY/O7H/HeBf1UZnXVeBt70DmAHTx6Fwt9Z33k5eDkPBL6CLdY8XJ88gLYO/nqko+5dnr3/47c1PHrlaEvjJMw1M6vcedFWZyjpUayHq1lHj9ZwVFaUEVCWeY2l4jln03BSHB8W95xCnxPply7p4j3U8k/HolyG5wKKWEljjw2oFvhFICoi6B9U95BJnPmjyVlG2Wi350BT2We4QhvUcOnSzEMOsGjFDC8lCCWrzvL12NjAC5KuzAhJ71y+i8jD3eb51F3YktQINAU5AiMgBr2oqWc51rXluORIsAkpIK7EOKPImg7AQNAfojoEuYZsOSjTliTrmaEZdxJHDQD01otLkDpPaqY7bK/R5CbKMQcFxOpjHG9j/FOWpYYgTuF5r9K8kVAqBr49vBmJFOSqSmqIKLz5CRYN+YFjqeoE79CsY8sJfPLsZ4xF2J7A+JagEZWHrBvBSnWK9usaXUyuIJIx7s0ugOJoMyCf7cp25rS82dy5CigQOfgVnn3tzghh4gJjUPdXVX0nVX56vhJt7PbP4NohIFT5oryxd3bvA3mMe+lFFUw9fcuAYJAWgo2HXQwIPQI8mmBPHIzUDItfYetNFueje0QIssDUYTk8RrED2nmd0ewfSgOJfEnhQ/aXPSYAtLisDcJgLwF9+H+JbBHtNB2t57nOh34nW4gWMM+U1XAjexE0EN58Zir+AlLgWlSHRCwE5QyMaNLuei893ek2ys9N5b/S8xX4Qrw4aM1SkrsnHROQnUY88agZGGcchdnKHfWluHzFSRqgydKuFvEMC22S+Ck3oEzegpBhjh+1w98AQ3M0QookQUgqJ1atQwJ4dGBJBdivCdQHGig6cLjAFOlksFgfmCYsK34ZrRQmjYk25ZTrOGjC7yLC6TO+jtWS8ycykktg4HKy30zT57/Svm5EcxT0sEZoo0slIltYD82Y6GNnSFyVrrN/v6eJby9lEfWSyFHZgqe1yQcIQRe7baU1ZS0NvtZ/Ie9huMOlUjfR774A0bbd7p/+kfE4Ol58qCl/oOESRyVrGOn092QliLZ1n3/Uoczq15xZ0GMPtcQm8e9Bj1BgnSbR9EEflH0QvA3EbKe4QsmtRwk8R1ZWg/RTxnK0aYB5Cnk+9nqECFPqI21Pw/ugebk3kQdR9WYsMtk7HGlZ1q3cpWCSmCvbeShgFFSQ1nAwFdtyNLmUpdscmLVUdyXaNn1OVGzmvek3VFvuVyD7q8GgcX+nhe6t63QwTJbl2dbqpifm4wtzXFURW1lv9UtodJ3Z6g0Y8095cm8Mdvbvv6xovtdMosLa0AesLDAeQOuVYFkEloaAEL7zRm01NQuZUY4t0ndf1JnUbF+hS/h9QZgP6HPYDSTMDmgavvP2eUyuvXQDDUfqpp8lG6ya/7U9V6NtDhbHWBEIVQfgWYIW6dju7mJuzT7DYYHWAsaYumzqstyXI0MCP40FIOWWrc7NJf0lwY+pnnfmbot4gvDEIgA7kj1ZeDaVhSC+Tjib7twAkVfO6biMrB0uHAItI0ZtwOLu14Y4/W7cpmh0q8g0ExObR7Y0GI0Qid9/hJUIlging0L31Dpz/+Ir9ot/SVUJrJ167SUWn4B5j8Z2nYIm+70EGoE8jVniMg6UF4zH4AjGOibtomFO9gKTYWYJqHQVeE+Jpw8VbUJ4/7MXb45dYpBm8dkSvTnyCes/1ezS9uAmH/knmBxz2Mftcv1n4S5S7MKcqUbNvxXbo0YqytgN2BBpzvA+B+tjCyE5H6MTbUG7ok2zt2M2dD+L62m/KGBPvWx/UhSDQ298JX8ostzWGOdenp+yf31oS7DrhUUQ/brJSubhE3KeqDmH55tXqxERG+mt/Le5jCdFvjRMcn9z0cUHV5EZ8uozD0Fras3pgMOaSbMZcU61DX+/WTcKDRaFxNXdEgBBfT6LR29xQgGOuQYlbPsD7BMcnT58NqbQgHfzdXxDj+NIa3yvIWAADazq4Fhz7/4DGAqBJBJDZD0lgraoInB94JnNeOXP9AB1MOzXPyrRnCYRcY1l/Yk0GyszxF4jGVxWG4ynPljFniMWnYyy0wxN6Kzdp0YMXpledSc4GwjT1SH0N2n517dD14DghSdr0NDe6q76x+9JGmK76Wveka7ujfj+E3fBKdxq9RO5kajr297p7d28PMoJv12XuzMr03YCNxejMTB86mp1wYfNPwCn8ZyNres1PZUfHZhNAAGNc42dOrTKGMZlOp732qKzdidTf7rMbLmjXZA8iwxkS1hZkd6Indq+Pk6XSaZSyo3UdxdplXlPq7C3aa4P5ycOONlJOxwsDcaKDBnidjSl0vnxPzMEleMFfeNqI86qC8NSBwKPm3ZnxWBi3ZW0fQiXMFcvImEEwyHiCNvHREEN/NcbA/wZiukh3OtDr8Ka90KAD/3dgo386vNE/8z3yQeihvzbsdFc99pjfaTI9JC2LInW7W2/eIZJu0DeWz7htoQQdq+1S7d267bRGnwvFHK09bS5VbOpouhE2+9qBR2CX+0KTt+2nAvg2gAZh4HE7ecZ0QLi3ZzwnbR2Bf1/I3L3Sx6dDmnxAzz5CjV12Dk4PSp5TOe3pFwYEG1/xTX8kn/jTMycOBWHhh/yjX+YrrAIPM6r2rP7oBawFYODKHJ20IYKROVQZfYiDod7FmcMeT+coEBW6wwNoDkFHk7nDJBzlYAlhgzYIUrCCU2dcvia7mzqv2GaAlPDEO+k/1Sg+qzoxnC5szxSOjqWofWJ7BuGBcymzUpLY8J1mpSXIV817URXK8abSVp2nmHqxXsioAtMVsfSho6ldd3gVyTTl/ipdjiVn84hbeyXzut1ZO4a/OOuSkG2bhHjOF+yCsX6QaXHbHunAxbXZ4jhP9Exuh416110rARzvhyZ4cDdaBg4JSh92MjmhfACUmJLZwTrjzmGPzyqhfPoQxPAghM5s2u+pkkbQ9wpp+6nVafu9Gn3JxiAfqYT+qGHwURV6sqko5VvUVkP8FHYbqKSiBidBX0fMAYIhBcPiI8RDW+05bDdIWdtAvONPR2ijVF0ltFnOo/9+lD2Kw0c/Pnr16KcQ0uXH+wCUkrt9+Nlni/YnML5qlpBuurv+wPSasy9RhTklmENv0bvf1u8af2q51c6ZWjF8z/tSf2kES/D1t2x2/Nh9a+EOLPkAkyDAuoHJds/ejj2A9jpdYjp2m1U03krjSHaS1LYTbY5yYeRE+KY/ZvOjotxiD5Kl5ezzpBi+BZTleHpSAIxPsXpX+JioOoNwYcSHLMfuU09CGS5lNvrkTekN+Wgz4f7cY2A8SBvV8FDyQGPZvPuAMiRvGNCJd9e4iTk73hPOdaMMxW4c0YRRVg88gNW2D2FusX8C8z1joJN4/clqAKGKvsJAZVU2+IAK0r0EuvzfRVnMmZERSP7OoZfCMeBHALmMyBUWz7iKpAzM+QBzSDDAwwtWbIzvoHd30tggxmZIGen7dxUEPi6ahDeFTv/8FvqdZyUHWNK+6reLK0sJHkhpAAweM8d/kKUzDHHMN5H+b7J8Ab/ug17WvG+4uAqfn7+4PHt3/py+kXnYzWbIHLEoImJcFR5UxUgU1XlC7R/MknbXn3gDWzL2UWUYgxhTk6u8ABELzELGJ9t1RksJB0wfEElYRoBrwp0L9PZBeB3cehMfFurkqLKODIzO0Y4CYuzZlXPM1rXn5/JSQ+OfgMNeclEWY9XV+I3vL89fPz8Hv+EcEewdHUEQBD51e0THMJqSHlLBdUGXV7vfCQxkjx+x9soXAsd+uSX9OzrSE8jcMWuCG1iWM4H7mE0fHcFgxywa7vS6HTq64+obTVYLwjToG2+KnPamncDaJ1Sc0IO7e024LUXoRnM3SboTNC7xmKLv7pEJrBfWSxp6G9V9b+xj9Q62ZQ4xRkkVc/3GwVSbEMHaS0AmaAuefmO/zAadqu/RQTv/kzt6F1IypZQ8WwXhQGyMFjrADz0Brfd0tQC/OJh6NTv0Qs4Znc+B3JxnwjHnjWCK9vARRAOoMQqRAFJ6PPU7PPiD30WiULC8iW8UFnvySL2c66Oni8XidLcIrZtns1l4/vrlxevz9rD+bIaWsxJ1KOgzZnf8jfIqLZY8ZWbYzHDVUtkpVfTkz3t6HXAaZlpi/w+E4NOk", "blender_rig.py": "eNrFfWt32ziy4Hf9Cg5zZptMJEaSH0krUc44iRLnjhN7bPdsun18eCiJkhhTJJukZMu53t++9QBA8CHbSc/e7TMTiyRQKBQKVQWgqmCa5tkkDZLcmPrG29CPpn5qWLtO3zg6PzNiY+llRrTy17Ft/Lnyjcw3/G/+ZJV7RhZExtqPci/yBq2WAf+NRfXO2OjkRtfonMhXbhrMnWRjdDrwP6i3jKd+GDvzEEp24lVuTLw08QHoheM4l63WoTfxBwCz5xiflkmcwpfQA1xC+Df3b/JVGtx6U6+Nb1eRT3+TOPKxE0kAOEaGHxrZChppE2rGBDBNoYqxMQiWFwIQqORPA3hp9ZyXXWMJMFIAMfMneWw7ULHvGO/9HB6xTmYkaQwlJgG0lEG50Jis/DSJDWviQScBnUW8HKdx1gZUI/wDuGQ2I7AMpgHQIiZUAU4g+tMGjCapj/0w/AxoHPp5bEziyIgQGDVkfA5uvGXMgKwlPQBFB4dBAo0AyWwj8aA3NEJBuvYiQteLgqWnsBVABD1+i4J804Y/0HQIKIihx07vOMaJn8XZAEd74oWTVQgAsxiQASQ9wA2RT31Cnyg68VOirbWOb/zQRuy5FSDREroDvZ4SKCKvYrMNNgCjEmWzwE9hzLRRRrQc4yALGFAUGzN8H9EQefPVNx/ojE1ju4sV4GssgU2h/yvjNiYsl0mILGVB3SwOBfWwA2MvQt6BEjHQLfPGwA5RDiPlGGeB4cFAeFnA7cGQehtAJku8EHoBjTOYaZAB40+ADoAzIEIYZEi9Xcd4h8OpE99K/STOYmjSWwaRl2LbaerD38wLV1OPf+T4d+wFoZcS8+05xuiGmf/j0VsGOAUgH95+bRtvj0Zf3gPjAZ/whIDx9peJByyQA0kyu20cv/0vMdhn50fEi7fIw0A/wrZtHB2/z4wwmPvEsusgg0/X/hioug6mPndn3zFOfWKutZ/mwQR6gxM/QYIjYRi/yQoICGRQZXgsisFxWqZptgKay4aXzgFi5svnbxnwi/i99PKF/B1n8le2UT/zYOm35MM42cif0WoJAgalVdKapfGSIK3yIMwMUeKzl6fBTdv418rL/TSCoWkb/8apnrZa511jSKAd/MeyW60WyAEY2rn11LMHRMYkDaLcmpkXMPWM71pho2Ocdwf7Tm92l12abeMpsM0sXGWL4Xm68hHWExBzf/E/pFrWgn/WgCnQw8GfF/KHE8CMurHMTse0jWdGb3BpBDMDHw0YLVkK5iPMuIvLlpcAEDkMzkE6Xy1BPp7gUwqd9xLHm05dT7xHsEEEHZv6WT40gyiB36n/5ypI/anoY0MVkOuPKgfkJNgzbxUC+MUKhScwoCm4V/tvsoiRvYYXeinDBPabpiuQ5TE+RUE0X0WxednY2sIP5gtELN8k/nAWxl5eNA5qoBnFOAee2VKp63TrmC78MBma8yCNDZieMH1AvM1BSMIk8UIgB+BKsvAPs7FBmoA6Vd6x5LintAtSSmIIjKp1qvuy21gt81BENlfabSYECBrQ+KDVm1tqrBPGcdLcyE5j+ZmX5Y3Fu21B1gRYaeyBgqKOozDNY6QMzBDk68QhvkagmYV8b7dAmC29K1D2Kb3JHGDONowMiHE3vhK8+enLh2Oo/91EjhzQhHPgJ7CUh8Ixg3cXl3etVgbGhA8FQfo4IIBR/jr0rrUOK6/XgX/tht7GT7mWk7KanSWI6U73L8sGooifZiSuYH6C+TDJXVBdlpBaM+CzmIQAtR+PwXzLs4Fi2NgRdTI/tz54ICCk8PMmebAGtrdAVcT5AhoZWraAWm5JNXSDDXHZooUbvQWmdK3h4vU6lDg6hAASOhYYzSardO1nbjyz4JtABRTLO9D1Xh4AW5LhJO0LtGKxIEh6R1S1QWntOrvPnu85N2i2gfq3HVRNCClPNwXWqY+8DuoxyxGGAkAF/JuJDybzQQ4aZbzK/VGaxmlRF6Bm3EcwbYcocCWBiBWQSHNQ01AbQQN/0WucIxeXdgEGK2TQQoIVqIhDjxptZbHJWAdKpVAuLrwo8sOxN6/BVsMPGD4bcjcn43IvBQmgiKA/qvfMjRILRkjAWtL4jL85Uy/36M0khjdR4oA1km8sGAoLFKU0DGzjqbEDM5omNxQiUbq/awtYqpwDnfK9ycKFLlnmBEXhJLYL+JMYZlK28BLf6vTaxg5/+swte2CNbhBHmPOo8t3rOA2npT4BkH8Yny8GgMtg59I5B53JTzuXf31Ksr3hpS2UBCD7nOslYOtN3ZmH1sYGOT4HJZVZK5BSRCchgf6TkgUK4dBkIAzzhZMlYYD1WPqBCrcvepdOGF+TwgdbAYsDC1kmrspQjcLffGaKYZY94a653CIWsGZB6GMLQwW45YcC3nAIYGAczDIQIAe8dBnWwwBm4xvzHizgczOMzOdaqReA1XO2yXJ/OboJgJ+AuYAzYoNWBjRW03hgmMAE0CiIvyWwlo9kvIi3yE+0rmIHuZhw/Dw6OzQvkYxRDDYsVd/WOKyJvHSyCNbUfh7giNOaJwMVplrTYRSC2C6RgXFBZQea052EvpdaNLHMd0ejg1P3n6PRiXt+evDl7MPx6WdUkBIQQ7/oXrZFQxdgMRIb8HzFV7bxBuzIpga/xUEETANPQKKavMb3TuSRYDA/00LfLPpFgqZESsFhSFAjyIiA8KWQVNg4ChfVDKw84zVppWnsrqIwiKQCLwgF8qlVwZoXmjDyrpckIUimeOKhaqC6YKeiiVc8ZrBa9aX9/sT4ghwTBrdeOhB7DG25pRDzMh8NvD/QRCntLAB9A2gXlo6e0fkdCcxWBdmT3EcpPAsELV6pOKcCJQuXMg40FUABSwMAi7xdmKlgQ9otEoslAd1awgJnecPSEqxHq2u36ad3Az9bC/iAP5c3F/1LWMIsI/jbNnp+Zx8EEa8PMofNZeO5sWhNYO002cAHC4p2L2G2QNXupQ0f+2162RMve/yytb1rZ0heC1aduzYKYn55jqVC7rLVweY6kw38Q5jZtl3AWyVTHObaGINGmPputozBBoF1meBnVS2Jw808joC1Xxu7XfyPV0RfYJXeRMBDeEMaCtbZF4O20b8k4tlsKkospM4altpSGg9X/bA6HzZjUl45MO+4y2Ear6KpdYiazW7hKtQkEQFyGVu+MCV0E0aseJBfqUH6xL+IhT/jqjfwQjBsM38OzaRyR2/t0/Lk3QZESAYDN4K1uw9/aUfAz2jyZmFM+kFoVYLk4ksho5a4KoVH9VFO6qUB7G8sHVR1EUiDrGzfRCTsHPzi5qnvO5UyAkpUyNq3Z+8/uCenn768+3RyNHpv1m2akh2n/xehaljl2YV5Gq/mi8jPgEqOWF64ay9codDqOvt7terC4hvRH2DQ5gbI8Psp+wGkSu5PcL+ItspoX1HY9Lj/6Mr1LnBi2zgUMhNmCMyPW5rhyJ8oz/lXT/3qs/H5xBiBtPJBUqWRNzCSVZTz5i6wXExbOrzv5qOlacxgJn7zhIQLjbH3LUayHBIoNkmpTQutrXFm3eCMgnVwfw/su0Pb+F+GdUtv9vvaizfwoscvGKkJiLHJQk0yAiznGaoYNYPpi82TtevsvkQYksFQYVjwkgAbr4cS6mscyT1GqBgt1aQEIz88MZarDLfyviFpYG1ZbPYaRHwPVCVIYIK0CBKXyM7QnmHfdwkpQewT0Ahs9Pt+zrRignT3sRh3P5xh16HABf1DA4hE4q9p09fX8uvsxg0V4WAUCLfICmdcsCBdOFN06/6q6AbV04bqaa16Kqt3SvXz2Hc3CoBEsFcdN/xQ1OexZwL6frgFAAnYBgDQ/o5O4oNosqB9ZdxQJ7Yu7ezzB96YZyMG5RBzLbKiGLqOGhPFtfxFsGjoz90bYpfuyz2d7RA7BKnx1phsdHh3Qf+o8XS6vWLQ9ephZUEmG6sOyziUw1LCCWmN5KLnNrbD3I4/e8SOtiTWeZxmYOJ6RDM+HUl8+M3TEKy9XCNN13nRrczjFy/Vi/KU7+9r8zlblNHHcgRcoC8H9ylUfNmVZKASapB7PUVlAU52Ex+ply+LXnLrsvQtDdTL3p7Gp9BvjdEKmiq8ehq3l1HpSup9xtOcARFNyc3Q/wZvDY83uUOwAw2UliSfuofGBv7+uncoNktmBqzEp1YWTH1dFpVovlul+a+FMMWK8HADBYkozwqurXHVBAem3y1zllj28na3JeFZVWBtSTAk7h4jBC9pW6hoB9DGnXBaac9pZBjcRLA8AqQxmLAwB1PiULAujQlrj4ugQD2CpQu9A5hgz3vhHOwAsBYnMD9z3MrwboJs2BNaZq88mxAaAoCR9dDIbW3pNHNAgiZr25APPf2BrMy2gVY/D/0ibBvxFU5rGr8ew16k+DaVbzvitdRFWAFlDRTRBpstM7GReOnAGgTW65Y5Tr1bUDqJPxd7w0L30+HbKvNSUD9mo61R/GeWziV9PJma0laxRoRpMMmtwyFYlCzehvynzapsSP+2SasM8R/6meLPtL63TcJ/SP+2hSQf8p82y6ahkEjIXkOatjUYOFvx061iuaFiPaQq4IHEp5/pcJHipuTb4y+jM9pPK+A8wfM9PCdFM3vs39K5Hp7bJd40VVuZ0npyx0giSwrtQ9xlBoDTC/MQzWX4K5Awmb1OcA1bnLfyQCxuuQrRTBTMkAkvFtAb+PpM6Vr12HtZeuy/KFg48idXUnLtawrSm8q3vxby7C02o4hpmXgCDGa/1cWeUAd4XE2ehax47KJAJjgeVzza2sMyz5Ig8s16Qe25j8/cYr1qz6yX1Z53qC43Uq/cN+uF5TOTR9Xulat/ga9mQ2H5gsmoavfLtQ/hq9lQWL4AGSx1DbdUq30eJ+4o0oFodeSrntOVUKhFAaXYFUb5Cch7S585njbirB6UP/JneBaC3EaTwiTS4JaneSrOseS3FL5p8iZDI7ci76EozkZkdcEtOAPNS004T0JvXa8oba9KPZ3HChDXaUDmRIaGsUX96cCDUPv7qpwfjuNrVY5rFQX34KXEoktqn9U6/NTaeoub1xcl6WLR3tMzGPFFvArB/MO9Y+gViqNGPihVOkiXJpck9HhUStC21fwAa2WuLWpSjwoA+G1b3UOPeEjUIJIVpKO3TJSdApxsTgNZqMTZDYsoFOUmHQXTQIKZwbYNf0rNosaClHKdX0iSS4YB6UX28s5eZci96Cr0tfqzG+ZLUgrmJdfaFcMnzdSi+lXkY20LkXjGwGgDqcwCnS08MIaFa71tUk+SRfdV071q21Bua+V2E3vfx3K/JUc+HmYv0GbBXjUIzFIFLs4lqd/FADOs7dwW4+QXdZAERc376p3H/lsvQ0nPdaCnOk8B0BpDCQviwjqBMspnwlrY6meOQgm/JmRJJ8Wums2bPEARaAqUMkq2t5dCK/8pj+2FWta3NvBUT53qlzZRB6jgV8sINf3KD8O4rPnxgA+Uf+5l6lyvtEdS2xnZAGttemqFsBFL2HbxzIsWtjY26Cgyww13H88PsV1mjSN4D2A6UIKXm/F0Ixf+ZN5rVuv4tmmRBxUYrWI9gu/UcmS/MANuMh1A4qe4UxOEvlp0ye0GktqXYvnVNl50GbalfbUdL9pYxfoL3h0VtobYqFl4MHDwSD0U5sxR2xCPJA+Kxz352GVjhwoLYwc4jQ5WpOWiDI/CjJAi+lJ43fiokGiLVurLoM37hn60WsLyPveJvTNN+3nanIbxJfxhwQFcf6tLjsZS6D9DMpN+AaF4ZQjfBJFkv7iXJYhvpWFvAc8DR47b1AGthOhPpKw8ZpINtCPgkogS9FoUnIJl5eYYLnQKLsEvavy0XYoCGWknaZ2td6Lpc48/L+iz0p12FbwwpB6oX/qsgLJhVQN57gVhBSRzV/flNpR7+JL5U65chfS1K6ZWFlVMrIpVZVdO4WcgQmYRLipSriipJzjd/ADrtNzUzQvDEvgKCptvPRoCiVF5fZ7HDer3JmtTy6JTNbUp3Eq3V5QzvFxlHOf3Ven2G+oUAzMzv2fR3fdZdMe6CzDHQ64pEecRtc4W5EpGNQATaLUE0C6tWt/+9cN4D496V6nfgh9SjqhjuGBu/G1YeIypk0j1FQ8cClezYszeo33VuDNfiAKxWq2sPd/blS0BffEO9srQ+H41MCw+AFrjARCJoiyIyOPUx3ckEMSMpwPVpZdYobccTz3jz4HBdf/Eum1jbVcGpek/5PErKIvM/d4Jcn+ZWfYdu7qoc3S9U82qmzdNPDzhEt4KdNglhyBzIv/aMkfSy1nsUPDA1I55K2XbDFfs9dEBMqh7dCBC9x5ZCQ+DLQBolw/O1ZvKeSH6oZP/Ef4YmqP3n84FUj7qhu93Sm6gCQMcm4dKCBApBiVdQhg6oM5zJgv1IdL0jYOiGvjeyUG6IW8QSH0HzxLfOqKw7YCEn+cLUA89v7NblhoKDpetrJjYetXVnZPGZCp3izXY+CJCnhs/vqNo4HlpGRECIxwRABo8wu/LhiJ4CjiJI1inY7liX/H+cTl++1+jd3JkmMjTIEtCb+PysSDI3fNP7/5pSn5yskV87QaRO0O5DN9pS6+Yc+yuTbMNFSj1UOyt41mrr3Od8gIsSrYNCUFtsP+0fEJP6kxtE2dXgPR44wr/ct/KoX0/17QFGLWnwh19wJWNbOWtffbs1v3SaWM68+foV0kHJeyJLa1iub+ljrxFU8WmLNpUbQJJlmBbWoOSVcjnDKlNI0IcXz+4BWUzpmGH7sUw07EeuYc4oBiy6yBfWCZtoTQ4p6GvUxCt/PIZue7kxYzvoi9HWBZzk3IxnChNxTyct7jN7JX1MdeehEFiWbC0ge/oqOCNcV2KCwKoh4/kL/GrTbOtV4b8JwoEmJJ0yIBq5xL0qjculSESSyXJZCkDQeLL75W9cUTqT7UzXlQT4wWlgRUmV9aU9hQ05K6Ff95NYnXoK3TJ0lb1xtOnRl8rfkFl3jAuuDgCO+nK95NpsMzYQwbq9ZyXvNQm6+GyJGSujedD49rJVsuGqs+Qgr1+ibG+PWjfk6foHH32iWnJ28K/ceeg+kCKoNMgGOsAqfEzSWUAOIzKxA6mN0yaKI5u/TS2rnHovskzPPuie1lzvAwQT6hYZ971HD2brQs8KgnsS7mevL4IECbKkNPRydHBu1EhRN5RZI2MpqGoCxF/k1KUCU93dHGlWBqxPopvNsIHcxInG+GNS68doYmV20mtgPTTOsEn8xG6lapVm1BOH5nDXmgV9atVIl3PNUG2Bxj4I1T9qY9+ZybRhfzpZAXSBojkv4+/jo5M+ZZijdwsuMVvh8DDvZd4ZFd35F3COHgJoBLQZr8stN2jQ7nv1rUSISwcyOTjEFooZq5Yk2nEKbyO3hi9PnodFU1Nt5LjvT9BWrwfvfv0+eB8ZGoTfOmk6CQFNRkc9H1bkyUvukd0Zap3hXQh8xf6vJHCa29tSvc2qhpfoDqoyqUtl73odq/r5dKQNTs4oj3A7o0Hp0CS305H7sFv58em3TicGAqkGa5Fk+XDzOZjuZnJUw1VKoetUTQYGBMD47t/J5qcBTBHcPlFko3EwRzFAVOnJHMUbwg8CsSeoAIG4k+EP1YUY8hZQ7hVoZfXwmzRRkF5mxWWHboVl3BbF7iV6kg1vXZ0XAW+so8wb1CeMtQ3UtBH67Lc20bl+yj9nWDeNcSXKUH36sEjULKM6MQ0LkXrFQaRWTqkrQ2DGjxaTzWNYLmnDSWkY+q8IGDNnNMkoVj9KUPUZEazRPSCbWrswks9s+iMNj8pVFO5/gl4heMHeV4Ch6XV6MrWo8w4wKFkvuGRhHDRbVC6woQpk6peVClg3eQpO++SoMuF7qqIxhMhicz3B+cHwsF5dGrKSkJsSMEq32InEA1XKEUleuAbvqKlREZFMooC+vfH0+PfTtz/Pfr08fD8zLyThQkIrLiTIJqT7jw++v3Tl/PR6Yn7ZXRwOjo7N2VRjutw19RvV8a7pGibmgdHR/eXm5IdZ345+Dwyf0QdAbyCqtu8p4k0dQ9qwTWFu7OxgRmxDMCOAv7ZFXGuNLPEfDUsCiV+zpHEzz/G0zjfNqD3r/KY0O7JARBTDOZ9WkFnKenyW6IfQ0Uyg+1FkRfDrrOHEYkJzKph3/5JM6DUMBHHzePcC7e3ToWGu/aD0CJBd5/iqrbDiydXLnvbD0XI1g8uosvirjZD7ZIAKMsGmuhzZjF02MelzUMTvpCMgsE4pjgthD6qPvj/lBxelAawMvk28CL4FfgYZbwRQPwsD1L+JCWbPeDw9EizlzFQXzIrqhSCvPQyAQXM7YkXxRRsji5ePTCo/i7cB1laZsEcsPPCIj7fYSKWwoydqyk6MMto43++P4enLTFS86AvLO/vcw7dHRhM0mJkalRkAXRNDvlkdlyIsN9Cu2sRVDWFiQMZbSzfYQhojPqdXarrY12l/sVJjbQy9OgtXiCATiMaDrFQhwpJrJSS7Rqvi4KvsSBqDlms4Jcr7AjTyioBslu1ZVa9NkMAKmSAoY7pRXDp4J5koB0yT52xF5IW1ha3Gbnn4iwpd4Htuxs+rCpve1QpXdvzkMOCKAPkR+5ruG1c97rQHmA6C3D315ExbyWM1Ka6aAeMa0sw1IXPgwirTTnO9iMsJ6PgAp2G3y4FS1xWR6MNHcPShE89GjGC79z7hoDDKlvjtiAvlAHr69KyuDCTJE6u5Ck3JQkyFcYTLrLlJ1utpsdQi2JxZM4Dw5KpDHCCryIMcci9W3SAxycuE8lUEhGsG0XMIW0ifucpOgDmfmiW3s8ntTlXYyFtluLBTwMLRflFddAv0TOh10Q1IIBLnSNiQV0OtcX8FMTgkRy2CWIF3+UZAOIyYetfWn5RbePuUk56Bvigz6MwIlDKY6oOjryjkEOKLGMotl2xcQ014mjtysHWwl9eSetEoCHWqgqcirZS+9Sg2NhSWTYbmgfi4AJxk2tOuQu9LMxMBPPXz6mKfCGtgy+fPp/x+QN7SiRjSzOsH3NAJTd2NOdF9N6At/r5Gmr2JM58TbOz6cjNXqcxv2gbT+FnVvhmYGiaCFYBUYUpdNCp/JtIi7NcobNG5w3mBRGpNgzae6Wv5W1o3H+V3ZO9G9ftCoGwcgP4TKceiLYM8qUGnDx2/1SNCkF/CmX/VX2JDI97p+hDQzM5jfWw+Eodca6CFWyqgbvBpzopP4MwJkEwtegbhrUJMo5XQTh1ORBdkHOW8uY6Et/Fk+Us9xMwSVXAZ+n8jGoWyyZlXtOaZuZd+S78SPU1DQ4scRTWpBWPi0mGcqn76t9FK8jOk7y+Niy4ZFA6T+IIRVdu0f3rtwNYCH35dPyFR/fK3/AOCXBWQfeZsjxhdTr38bRKEgTpYOOu8AW/uSyaQ5pYM7DN+AvuxPOvYrnNAcH4kgzfmd18XtHUmUqHCiZqYp6igowiLU7UNVcPaAa5m8fYyivYMCesYVaguYsoCaFbQYonyLh+OKDPFNycIVBNBvlWm6NysuyFenachgOZbQRae2H5sK90aNxILb3KIwcHmInHVth7ZgMypmCl4axOKyn4t9KooQmJ8Ta4xOBkv+h7GcTkpEsbU1Y0BinOJlX18+73d0ejM23n9/4gRLVeLYGm3BXUJZprOIfUdHv0CpgU0gX2D+0HTlqhg5DqZJJXHe5RyKDTAwM9o5hjTEcQsANUEsg3SaAiLhYoyt4XLs9t9SA9VrOF7vH5vnBr7lLZklfzExTXqzBG7FZCBVHUBW0UIgjeGVuAlgRzBBbyqLnG3jfa8vg/RudFX2TuEf4Q/ppC7whrEJxR31qEzi25LsPfNh0Pwq8b8eaGzgh3BDZUPa1UT6vVO/Cquf40vo5E8zd4aNeWCD0zSlHiL/p6hbRWId1a4Wvb+L1t/IHj3GMvAuHK3it+FqedhwseK0x/oE6xYRpnFpiI0dwNMSSef6b4E/N6DNHHrc15kbgA/4QCdk3jfy/xOHlrodvzgI0T9Qx4/d6W1OlI2IjuV9l+aFecYtnZS4emXjC4jiWoJwGmdgliWoOI6EjPbB3Fwjkca3eQCs3YVCuXXm6pfVdQPvQx5RHG7wCybeMKf13hr1kcU6aqRxGYXY91/KVjM7bvbaNkuZr2StZrple1raLG1baWau3odbaNSpxXhoQ8qImiGG7wFJuDWYGU2jY2GojiTQVG2gBDG6FFkGTuKrGmt7Z+CIRegYaH4YUgiTx04Hj2xwCTGYJRLWzn3w2M7gphaiobe1uoG0yp6S17yauGc1SsVt7AABp2wTQsl0Hr48zCsx4Q1U8NzYbhFHIw0UVwyJg99MVPjPbVyl5I917UH0zAImyIqUcHSmO7VIn8SosqMl7nDxVFoKOGrgR2pT76WmJ9SXWoxnEIC2qsSoqkoMS1F16VKIFnBkVjJTLQrOuIGPQzy6PY1OKhvgPTdV4g9iSS21gIcXf2barY+Amd64NS72T6CTEGO3uq7U7pCb1UH6YKx2YvyEUls5isnm2z4ym50y7qw1kdzWJsXorm7XqlfrVWX1TrsN9upVp9ZNJV9KMD80InR/FQH5geecxWiL+L49L85cFh2ddHpXgAaA+Pyc49Q7LzA0PC86v30pbD8ygyX3tr/7F05s4Kn8Nev1DoQn9WbI39CkOWlFzRgaru+11I2Y4Mp1A0eaTQQCrs2QpSd89+rIj6vS0jmB6k27fVMinR7YnRxRlpeHNvssDEhOSYjVxpeEFKTy86PWPiUUrc1J+sEj/1Uv0EJudwjUHNw00a0xY6cbG7NwzPc91/i7Le4iCpDAzIUpPyKgZ9SchTvfyeorKlwwybrwqbF2VsVmgw5hwOYhMyu3UUhGAiDBTqK0LariEEEDBGHb3oq2XtOpYlBuvtdu37mqivTzX0XzD6O9uJTTit7Aa8FaHvo3N3C50bRNcVhf3tA7uqX1f0P2XPbZM8YjL2Czsb0wvLeVly1W8SP6vkR2QLxUFcoQ+W0793ckzZqfXHpHe/JHhtlqXayw6/bVKze7WqpXedqo6qKlVNyLBeLb2A3ra3HK+oxY3zK/lF7irRr9Y68KWjfXlYH6jJKzFgVdD/AVXwe1vm4BHkEAphrxmJZkn6u6arawK4POalrU/zlNJgA4gXwJZoaop9T5G+sVxYZrxtGztQGs2xxlKUSBsK9WE4wTJoKnPGSbah0O7LNum1LaVyLgSQUIg3FXpL6bkFJGLmplInuON3jki12ewW/ezZtchH3h959PbI18aVOS/HK6YrLLiWyRAlLggM9Hx4xKRbkOMLrY4ooGmAoFkPU/jSQMY8ceUq8wuNLarCd156iapJUMRMcf270nT/fteqHCJSxsBk0bwvmlxEMthVsWhUxNTSinOZMF/D67gaByPqUzxSGYAIUSLZdlU1+mIy/MByKE8VClcrpoqIXuPZtfMjk+trWwYsPzi5GhdsW4ZWrvUYs0ETmv1CCNSlGofpyXoy7cJXbSG2pR71a9C0gOupscFl291/UmDsP1JgiJipfKDmDKh/StiAK2eOE1vKrJTFnvGPxhWdHJ+NCv/DB8/uapunDYFYNQnyox5A4my2OHXkaBg8lKHdXu34VSuEh6rFSS0XlEINGiJkakeVS/2QAilJ9Wr7EC/6ykMTDy8jTjhYKSvPNeVgU5r74kGUZ3/JCPPdguxIFaJi8DC8YKhvaj/6TOyJllx69xmmUgzmEXrne0bqRUXeMY9S+dV34wNKq+1Q4kQ6Ud/aqMsZF5uOKh6oMyyakBELj9jiF4MQ/fWTbP9GpD8+/u1cphHFHNKYpDfTT7W5oHUVRFMZXpsvBOOUqDbTz90IzAVWutTyG4+9zMehtAiGKszhXDcixy9wbnP5R7mQb3VgRlzudMdw1fLo9PT4VBIkIgS4t+JWCDQUkGOmfjaBdUvMMR+SgLjZh9mc8mASg20dLJPUz5CttsklNQOlBUKzQ/2+f3LokZXFEWq3mN2hy4i5wpXU2p4LHn0pGrO7b8Ncxe01VdqWFl66b5QlY2MC+fE36Eai8QvJL2BQkGV8GZEr7leh9Ney0y48yG42EYDPuDGgivJrqTTUSVsMosuppofmx6O3MPh4Xs9dVJmGW9V1A1Ujl1727VSgNqtEJCcWL3Jg2DhOp9l9kNilNCvXVPE6Q3P09eT49Nx8eHiuruXiSOIoRVAFODq9i+iu7aedUlPJOkVC8adPr64VrELMCf/Xd+efjr+cme1HebUZ2ihMgPJ430UQzSu4FdKvetB6vkmqqf0bHYObO4FMJEScCXwkGQzYANkKpB0USNzZ+OZetsSbszADumJJzHf+EEs2IoYVS1KUWRUBNnKmgRdzhL4342BmwY1a2p0rn/o0rLBMu/iEfikoFoT1VWXUcrko9Fy+30AyfvFdDCAIxRyErnvlb+Q4tjVVli8Em7w7PvkdEzEtx2Ag8f1IvuRSGY1O2Yg5nujLb+hWXQiimyDDFq+9dDo0O3+YHGUJq/ChiWAxTk14nLsM4AP6K2qYpDDS6Ybo5lKAJtXL/EmMmflKH76aOp/gSLcNHhIaGZG6mVLzsirwMB2kdleYuPuLb/cqqYnE5RTt25lLahdK0q84DB5+RrZvE7wNqf9FdzUmhLeFACHoMHICXKN4a5jlhVTjka4xxD0gSOC6ygFDcqDgAjFWX0YfD84//Xvk/nEvrFWiBl0fW6Rx2+CeUodp/md5eO8QqeHcmTpQthglePhPj1JjuAUMGTTVMGQZ5gj42SG7h+bzMB5jcnHKwN/rdtUu5wO3rlSEHs5SxLyG8sNKuP5fI07F0CKBlnhGwS1QQzRzT+jStQFdO0cXsGm+yNMYVy2USEp6JT/H3BFg+GGhBGVdZm819hI3uW3mHL7pTRoycqfA5df6NgoDKAcO8/v7IodFCRU6TGDNihPe1vjh2xKQwuNJDxwWH5WfrvL6VNEUAkP1cq4SS1XDKKBkyT+78OUH+nO4N8Z6Z7WbarRbaoIo3+nbPxoE0OTaTbgKZxxZAkPiN0OxFXEzMG62eu4DxjLmA9c+cwagUWZLIDZZOBSsBp2hTGpLlfUe3ls7Nff+Svj9vCn+vpS1pWiYXNeI11xyOearZyxzYNoXnd5lPUsDYCCQAkTolOPFntgpLLfnAK/MOIMIRkoODevppJZ2QbtpQHeIVUIio2zBsPaBxTdtM04bbyCgbYjp9lsHKn6gmbpZALPrGe8QwYarBRoR1kdNLCqX5QHBGGvBbzLcupIKgVif2ONSi/WFsopBy6kL6JO6x4HDVTiKgvI5g8ZY4wxAxlwDL8WrKNfTParA/tv7Isy3pNbRC7OQLpQX24lD82x0VGxVNVRAgQnts/WF8fGnnw6OzMciU9oIY9ORgz4fuO+H0paQESriH4Tss8sBQrxtjaVqUiGuzJaG+BgpWKtFL+JywP1F97I6gkUDspDICnfpKM//op1yiF5VXjTYFA/1L/6x5TtB0HNqbF9OaSo8uW1eWz9iQdi0xqmsrWvJuXiAZaIgQvlxxPiBa4tahQ1ryRbbUl+3SdGTMXF0/H6AVkLGib4pCz3e3gp/rWUACxLtAuGNHlDCSwSA8G3lz/GuV77ldeKHmFrEzxy2KMN4alFmidJOXBjXjYQwvs9ACBUPz0zA2f2OcoVTVjzF8Er77jGZRsK4EkasXiwxSO5iSeRfUihBXBgSdOdMIbdV2E6RbZmqAJCBtlenGSJinIQIpgQdpe/3ZOfQMnPQX/lS6KzQSzLUMQE5JEOXdAX1Mwk67t1NW/YbMb8npklUKwc2bZEFYfw/uMHXtMMG//yPbLJt2yqrbetv6/XP7HZt3VH7wX2vbdtn93nCPLSpxbtHEbDCprw3DmObC4MH4yamLqeDaRvf7+yL5n13KVfD+L68NU1iNGySoy03mgmI9Zu26BordIV2c2/OWTwpHQCi2nP3upSzkz3T8VXf7e2pNJ0wvgj6KdQ33gyN3p6ew8dNmlfwM7mE/44t3on1WHmELPykziHTIeKXAHsDzIEQyPTCRn3AS8p/y3u6B4aHV7LTLV3OIl+GhkV3F4krybE9vO7URx1BZ1JSIQhxTxXlcpDqDzG07/XfpvGE5Ce+e/OavoQgsoamn5nw7HvTN6+XeLU6u7PBxFvls85L802LX1NqDxMvqMQumhR6BAu5oXkdTKFvU38NBmmHHtpBFOQwuzpiZY0w8iAP/TfczYMTY+f96+f8qvU6m6RBkvPVwEjdVeibRpZOhuYiz2Gt9fz5ZBo53zIge7BOQeDlz6Nk+fwf8zieh/5zGo8OIuan/9hx9pzuc0ypUnqPGc4AAnT0ObeGzeYbaB6TMX9feuk8iAbdVzPoVGfmLQOYBhld79hZBa/G3uRqTvk3B096+72XvekrWqwMnvi+f9fSG/pOBBiAPlxfv+Jr9gYv++tFCUa/39/rT+5aMG/S7yLn4WAW+jev5l4yeJncvMKHznUKT/jPq8SbToNoDmCTm7vWeJXncfS9BHG2P/MVVrPZ7NU4Tqd+Cl3iHx10oltlBFxCg99Gbxdbw15jzq1Bb69owIlns1Iju7u7d0A/Itvr58wxSD7kEI0ERjCFYVyLIWw4hzEmHq5AO8hBaRxmhrfKY6SBgf6ReNtffN0JkLuyIN8MzZ5J8yoD/QYPDj5G6yCNI8yB2AG4c3gf+Su81QxHWEcGcJsGa7wqIANZP6bjZECPjtexLHx801gCZqkqIFn0TQtQznJjuR7CbKKrrTG6dhT6+PPt5tPU+mW5/sVuH2z/Ti1DkaPtRbDpX+xXojF8GroumFxnrvuqNVtFfHw9zsFybeftmf2dC44LiBwVKoBav/B4Asixg9vm78TMzeE5jiZhMLkazl4lYpn6bhGEU2tsvxJnx2Ng8DXGwo3WUOkIZhaojhSx9Ka/tC17+Ob7AayT4d3h+eej4S+/vLKw/Bp9qsahf6C033//98WljVcAj7zJwoqGb1rUh4N2BOIfhn/iW7+4v7R/MYA6BBbByNpfKLPfK3iFjGLZdzb871ULiaNAWhezdn4JNRHsEZBGQkFGnEGNVy01/V8/J84FNkZxKMOWERxaoRdNbNtGl4QwmHpTkH14x3rumZeFAXrVNviuXUvpHvSUQlM+Nva6fydNpHQQfPq82rClH4Pq+XspeTRopivOxBBWN6Gox2JD4UKcmuNFDmILBlewRpygrqxvIRZKBZExr00bD8NnC+1MfuFcpwGY5VhIDYsp2Q/Q/paBZJiulklmISq2re+YUgMuaDFc6dBD23gAD9J+B+Lq3KmvPDEwp0se4/1by8SD9Q+e9LcSd4yf79tad6hEsaFOj1aT/4E0iZCIAAyT7yhD+AfyB+FOurf2XS8DqzKIEJq+sqX228QxsCaTJiCuq5R981c9M9CY8+hmz6nP10vyrdFs2hXR/fQyuPVeYWKMGZ8+hVB2usL4IiR+ugrw5tvUJ+mbe5TNAcoAo+Iak+4TPfnt3qvVJVnExdU6Jo9xxyAHC65EKMbom4QXbnTeKLT4Zssa3poZJtagdPWpyHMWxRgTPS8lsfv/hj/TcRvCbEqRPxQU+hEElcvWKnHFB3morN8m7kdg7XADHIws1oPrUpsl6gDzzrLKZeX4zgeQeLM76AfcvTK5C+al/rG0syLuIz8+Of/0lUTqb+8PqlmCG696JQwcnEerHAcUaSQTRedbSoNmFSUzy96WO6nYD84JwylnVMT6oq7dfC9sc+HB1sXcFFkR17xFg41leSCav8HawLtq/dgdtqUkAY+QbFuY8SMwBtIM0fubYBSRwlDxTKkqeWb46tJp8dh6cII2pw99Yhx/ev/FsFYZBXYDNjiNCse9/jP7lXEMffoqZRvMzXC+imCCYuEpbeBrqZor3cT2Kd0G8Oboy6fPBx9H70dfjj+djcz7qmDEMHZgnqx01B9NZjWNszhc0aL+RhKM5Lmb+vcX3zyiOOijpUvXdavTt0IKckk0l3EDBziFYMGTut+bEhvOv7LjGf+ItMuSAgdaBhbD/agir1cT2HHophGs3lOwUNI8ANvFLLcEggPPp1zyDb8U3PUBsA8mpubgyt2iDVzgRnWhpwqevdYPsa7jNJyKfTIqYupOcfQV/bW5YvM50xjZ8rp6mkSZdegGEF4hCbjjuTw26jacFWHA2R47WLNP+J46Oyoq9uoVeyJe64lxtJpQRh5jhy8uzYos/5R7JIwnoHIw+wxmUWaL9F0IJgr6aPdxc6Sz0zZgoQw24K/dbtvo488Ht6DQbzuE2YauThZC6PShKsDY2evSpRSPgIArj9QT9+rtICp4mcU+IaHbv0f6+IW4iC7y5NAO5+hAU55HDq5I5jgRYM4/NSzKG+285JznWjGRVhr/qLfxtlsquK2jopm4IS0MvLH1FrXCKo2JvwpJrljlaxwO+UI8Cv0sQNuY6whoBNYopj+x2FEJfU7wA4Gy7EeeyqucaWcrMIxbtQM3dGgKcFPWBTMfXZemeJntrT9E1MiRXiA1lIFn8sod3FSqbe2KT8qFgFo1Ww8dKnMxBrzlmPc/e7z76GNdvO+TQhpk/GuvBuD+G+df7kmaVDpePhB+Yrzzlp640HriLd2pTi7eLJHJY6ig6AgVxZtEkDx7Xflu67UronKbKz7mzhUoWCpGqGCuOW+pu5fjI5ueXrDESpRMiy+ZaBsyJQK8b5hBloq/pPxbT0WlDr2exFn5tbyVC2H9yATT237cFCuiH1ivxqschtzSjtCwSV3LNiwpHNqcKvQfXQqnbjf596f3o2NTWjfSktlzbh7t3V+Fj+tOcW6CDXz48Plk9FHk1ndms2Xiz53iM37crXydwDyiRMaH/f36N8oplbt4Su9y8l0q+unjYbUR+uPikpdScZofj4/fqzJidUynxfkCSE0kzrTk8YqMzM/lFU3rnjWYDqkwcbY5+YulM64gCbZ092+J6zZIEDr76K1VABYBHvq5EL3TYlpKXBBxB9iHv7DUtNM5OWlK0c47GAfIXM+cvN+3m3OioetCKdJY/8rbJVYkIt564u6bMAY1gDOjZ8vs6fR+5kHHyezaKbngbjkL4bnxvegVAL7GMbpzlsmuvsBtnEiNy2Z1oiHv6aF67VqvnmHuInOCIVN0Z0+xTiY2UVuHFRcA3jMTmBdtPRB28pNRJNRK0SPFXw0MBR1FYZrjpungxw8jiygREV5Hxrr6/ZgoEc5+LFzuev3uFsYoMgAkwXqrsjkJ1nGOJiddGPpYP8JgrV1aDDK72VlQTZju9hlSqg1wdaQbVIc0cErFtqemo3oqe1yl2jbofZm8QM8v8CPt6MnZtiakQ4CNcWR2PaUvbzJPCgQSmNt5wxbGFXpL4g32oVTf5tEnzItv/rRIEpzWEblt7xU0YrbOgzTe2e/+vGhBAAZA+ElpgTX/X8oJgK8kBB0J5x7dLBp4xsmXjyWh8ZckxE/HkW2JJVO0bLKIGjf0HrCMaPfF/NEMiY8xiYCO2o6qboo0n1Pknssj4CTRvM42dFgC7B6E4TbGKUDwhSA1oH+NbwoeUawjT2P4JKRtyDMIcRgBJQQ4PnVhRzjCV3zI/Dnuk/AHvncyD5a+g/9YaDScc26Cew+b0mAO8mwWO3hiVDpv4m6pgyQLGwUR1zbQ3zHKh5jdJMLTXtfLJkEgfXnE+mqY5andolmNcc1x+UyKugHc+n8B7YFsjw==", "app.py": "eNrNXFtv3Mhyftev6NA4EKmlqNHNF53MCWRb9jpYrxVJuzjIrMDtIXtmaPG23aQuFvSYPAUJkMcgQfID8paH5Dn5J+cPJD8hVdXN68xoxmufIFrsDKfZrO6urvrq1rRlWe/SQsgJ/8RuxJjZbyUPo8xhoYiZUEUJP47gR1rIjNrSrBDjLLtiGQuyJGM5n0Yp38jLcRwFHNpS/L+QXImUM87g6looeJSVKSvKFEjA9as4K8NJzKXwLMvaiJI8kwWbxtm4uv6osrS6zlR1JXkaZkn9S1RXSgRSFHU3VY5zmQVC1S3FTApYWDqtG6JEbNQD06IZV3C1sTGRWcI8Zu7BaibR1DSKFFYrqltTUfi6ZWPj+Pt378/ZkI2sVzwBjkjLZdarTEpBV+c8LkNeXRb66iWPYn11JvJMZdblxunZyfnJBRK632Dw9wQYnoylOGJ2zlWmXJYFhRTCZQEHHrusELdFKblDva0znkdhZkHvvUOX7R0+ddnBAP7genDw3HF1r2OYAPY5gPb95wcuez7f5z2/jRLqdQh3Dnf3XLZ7qLsdDF48hW4PGye/P35/+t3Juf/63RnM2NrBnQdR2fm2TO9Knu6/3t7b4UrBzuyIW57ksfCB6FQoa2NjIxQT5k+iWCg7dI5o1EyGQhITkwyELfN5Cv3DzJvGY2STaZXR1JuMb5sWbxyLNGz10FyJPtXPEvn6r+oGAs6LKMi8bPyx9XTdnBRxqxl2XQoVZel+6Cm80yOaR+ITV9VcuQyiOObVT5izH6WTzEPBho3GJyYK15opL+fFzPuYRakdumzisEkm2YRFqWFINGFVJ3EbqULZ8884NclvgOakIaFAUkVoo255+DH3rLVlOY7TXguMN0FFx8eBIOgc/ZqA+IfqJipmtm15cTa13GpYUL5SYudqX1WZJFze2bhks7kcFkscALWxLf6xVAUIgsvuH/TgMegRMaSeysTa2rqvH9kkid902eafbTpH7oPWgK0t9p//XmkBu+e6K/zedB5Yfos3rRZBoX4pRSyKjLUIw95gb7vVNCsFaBsMNoAb+ofLcvxioPodkq2ncvPQ5jbQc3Ds1k0lpiWgl/KLrOCxWccDUy05mlhvMpnwI3a/mZRxEV3DZvNN3I+GTOuGw0SsBNvchd0uss2HzrTsRUNPkDyNSrO7MGx7d3zUXdKminrDGg77UTNsikMuHMZ0bgZqEz8DKRLyqLMB1FQxzjBES1Z7EhbBQQAaCIJjpKqWHI/nOZCxreOmE6yKfcNQ/bS0I6VRh8qlkfzOMHEW+iRcywf57sNraJkKmc0N0kEF4OUVMum6FljHIt28ctk16peeUjPipRcVIlG200zMuhYS4EgoX0WpjzLmS5FDb2C1VdFYMtGJdYoyecR2BwP2GzS9MWcJB2BiKpqWgnE085VOWF1E6wvTaPORmWxePrDqNqtbHaul3M0WgUVjW1sgxZn0ZkUSgxajc/GX704ZPoiEQMiYBl5mrAATKTam4FBM4adk1HJ85pkxkK03FT+MwOAQCDKjS2cph/7wD//y3//xd+z+5sEQMmhm/ZT+lJpNpUccg25TkQrJC2GDU5AWLhvzALYzFhO4BiyZwZeaCVF9+YTicEsk2bXwx1NoL0Te2PIu1yccOEiGHfoJAQiNXo+vL6dlFPI0EDSQy2ZCDycBUgpovI5CkfXoURvsE9BUZIOROKyv4GA1KwcCLsBjmkIvNZxK79RcgxgeVWJIi2WRYt+D5pBRoNVVLQ17JY8AHoDKiZSZtK3zckxylogU4BMkD9EKRXECel8IZmfgG3I2yz5ybCVsU061qzByvf5mDPwFtkL7gx5+RWlhow+ztbW/y7bZrn4a3DPo1jhqtrNwOUcNu56AxQSZ686mHheks0AjBdQ8lcdR4RMT7Pn9biyqHmmon9VySU1mgYW8a/FOVNRlmf5q+Vqsxj2pqwQNGQdC4FTStljElpDsCt66sraEWC2BMU/GIciJy5KjutWGn6FQwTAx0ChuA5EX7IS+AM7Rf2+JYeXoS1gmcrC+Ubd4uYTF+0DHdpZJL5hkQEuIge5FBQ8keEKNrDCSxpNDxKla8do059BWrQU86QWe3lqu3SJ2of5VXUGkVPRJLHqU/Q68eoB+stgo6ZoY7BNMDX/XyKn9TeXFMAtYWO2TV+oCLi/KL7l/lt5ny6FpWNNIZvtPB1btM1Jzbk9aJJpR52fZBl07txc5/w64wqy51ffvna5IPdYRb7Z9c93S8t17tGDS1EMv2SzVS/IDfHJEO/4pyq3LS3AC6nCmS6LrDUM/kCq0Lf/zz3/7r2AGR2AOI8nIGta2zh4DSKJDnFeNQfRf/5a6YO+cyyV2um2wd3RM60NAuIOzGt73+b7ZmF/0Vre2VlEFKSA+VE/BpgB84p6TRUCD/Lgod541XqTV5zfCEQKRU1nbMS+CmU2cNZBkwGaVwaKYBR9bYZquuYwAOtAqKetRswEbDXOAyAfDlEv0Kmr1iVytQSItE+Mc4NAtDahxLGI7AOap6WBADV1/ikpAPHYfdu6bDhXsaPNTzGDsCfmr4IOBTgJSIzKqQhqOTryUJ6KFdnddLZQtA4P0iKnDhrFD+uyCDi68cpikEfhuD2BKy6P6x79itbyNKRWUCBoMIoI//PXfL7gpR5tAdfOys9wVCD8/7j/9zdJx2whusEZvZ8fNy6aV2M1AhDPQWbODn9AuPx5L60yR9+GHC0yIYGDtIaMcB00zeLRKDC9kKbpTUKOjw8GlGRPEIyslWOVq1M7mGYOWq7KI4jkzl8lgVjdeJ+hvUEfvOpJFyWM/AfcA11N3Qimmx7ygDLkH932UeAQpmyxT62akfH6N6Sqw47YRNHS3Bq3NWqQxbWSHWMiwCCVeRmNhCy93HpD/Z8fvYYOuE69U4NjtkBs3OPJ2Jw/wYx6YgBT0pUC66TyAzm9fYrCVeLkANqbFw28o/Hx7+gMrFQ/50WJa9nS0ewlO43Q0uHT6o9/Tzf4onaC2IQU7CPE5Rj4YmwQ8mIFvScosPPpVy/dC2b496nMN9NoG56zCwjKKQ7+MKvlYglOnRsVz+hUolF3Ms+IvTxuGbczVcXhEsvuE327fRGExgzDxYDDIb9mfaKniwEGdhoSQh92jzzSVGcT4RxRDcblNxIDR9otBKKbuk8Px4XgycJ88Cw7CycT5LQvAmYZg/2YGYe1vASnTYht9FRjpeX6riVczQ9cCsfllnAVXyi6iAsyWdXzK3qMtZ/uvGUIkBHPFTCQCUZ8ulHeeTQobfDm0sf6shIciiAemWd+0mD/gxxD+d5DjIShFw3Mg+Z7LqzC7AfC1nrD+2D+lBNKIYCY0hRstF8NdGUtPrCBLADIi0CNKlr+PbnkCD7aSEmDZyBthdyAQtA0BycdSki9jyp+AjL55+Xv4fPvdS/j88PLP4fP84jv4JEj3UNF+ntPBU+fh5xbmVrtwwce2daIrAFbPkav6nGU3du9W+/arLC6T1FYBh408WNCxHR1B/3eYG7YBY0Q8tN5UweE4jqYcoCjiDu79XQ77m0dz6delXgsS9XG7htbZ25fHVhXSDPcHg4WudT3/4yCA0Ap4b1vvYTt0KHgEe5zzGFz6HQYoQzYaLrvxIrPWmxyz7ER8RO8bE3O44YZ2Dm4Q4BWuOAPjNoQgRIklHFxjR9p/qMUL+H2iR+6y+BHu7T5dwr3aMkO8umCcU+IZiz794n29sSgqXj4YKMdXGkynPOYH+ra7/TYmNGgIKoeBJQ0x87emyJLlb8117Yn5VQEFpvdaZjkB2aNjjnQqwqXkAuUZiJWYzdQ3dL6B7mCfVUswT2H/hkZNHJvoR5+2dfk43WseI6wvmevjYqg36ANwJjX5T9JVG65BCMtIyBALlcA4Ecy4Yy3msbhd6f+1y2Hk/eXpFEsro6PdvcuNJRFV2+T3/zBO0WUzZWM2MErzslDDEbEBwo9Kcz8K6FPHLmug2SseRyEP1wUXLHtVUn+GzoONSQLb1CrBvTXbY2qGuKzTkXUtOTiUY+uS/W7I9p+vL/rGu9RlykXE9kw+oyp3Op+hVxXTTqXQJSjL+RqgSqk1zaFz4K2Q9i54x7tYLoXZ012Qceo23K137hRLutbj0KazdV3SL/Zc9vRA09b3a+L7e5/DjDZHziD4iEtyNTA3SwbJ1tSdr8MkSjj2mGQqyvoblkN96tVgxflz12OW8wrLKSwhkVzF44K0uw2aI2t3sHeAeIVFcfzGqjfOS8s6uuUwW3gQAuFfOcOq+mbnt1+JwSYrDp2/L5Mx8ndv/6CWtnORYNbLwsyJCCKsZA9X2dUq866pvpqJ4Gqc3dqEGH3CDNw87aitYHeVYe6KAh4wcNmhNzB7P/AO6xHeQBeCLABqU0D4Ohyrc+a9BWKgXo/+F2VUcPTR0jBbLUmY4X6MWrXt6Be8O14ftE+q0KIVLIDHSK79ukgOBrMN4yNrViY8zYAEinhQ8lCWuQiRnpVG6bRMs1Wmed5Kt2nWzqWZ+9fZNe0QdcRn4AEmgsDses9r+Rk0AgS2BHkOoB8zO3FW7KKuanTob+8+R0DHj0pAXwx+LdS+jSTIk4pYAhdsBlsJyoNnppg97ZRNv5BPJBs9ZKODSrXJbg4rVYyqRQvjXi1czupBfFkhewOhB8isZ3tkB4l5BweDGkDRplcPfoaIrTRc683ZFKc6O3xgDkyhvda3azPUYKjxnNBC6kMLjwxUF74eQ4MfaYuwrMCwhLKY2rhINY2XZVEgFrylCrTE9IOGEfYNa3TMBfdIJH4UDi0IQOAnZlyGVjxdQH5hrP50iVzhuajScK1OlOgkOgY8VNe9Y3kZK862tswkt7a8FSB3wcfKXiO4pZTEsS6QWCtEP6HykZkq5mr2X9t9MAKmtc+C1IHW4d7nazYLwPhJn9JdQ0CjwTMACm/wHD/RtDnOessz1mH18mqfpbe6iyYd9WUr+koLOta1ttULCmJ+t3BF51iPJ3/0/8cWnVJ2bvWCqrMC8yv6IeU6x0fVPTpe9oVL+5VrW9OWXLeUiQDL7luLNbJqmhCA3AI6byv8W00mKwtdZtVk3sBlReW1APiSUzSjeJLIJF5IzWUwi64zxRRofpF9VioGBwPGlmkxtOjwHZgGALTO85ib53ke3/k6Vrax8LSAoXj6wsVzDei8mPh5hH3n8wNV/r96AiOOojesHswLZjydCrs9AZdV36PFpz4APnoVPDAyXhBHwZVdnW9axqPRH+Xc0x/vBNTnnoVa6o6MtFVxNfy6BrTcStddoyiulnO3kVXXmE69nqX0QWKDUoJjEdz5cZREBbpJN5m8EpKckfb9yrq3RKKDUt9lmD1v17adPmR1rPgpHZjnLClBnMwTDE90AruVx95jlQc91KrYQJYTzb77SLLbgmANi2933ByaY+bgIVITfb9gPKfYC3XP1SqJGVI1HFmUZrWadNibVh1/TZwb+78uMFoQ8Wys67U+FhKN/S/x3Oe4Ouc+mr2WLAYhsVx9AgJ5bIpocxT0kYe2vPR7gKiv3jczUwBntWiSBn7owAcA17jSHdoet+LKJd2C8Vw9sUeCh8/Rl2+p3h/xuK8ls3n+HQdYUQcfq+/+zz6LD2dNsZjjSzhRhlGUja8fgOZhfsaZo9+wyRxQcOn0jauHXra4MwF8oFxj7yzI/Np+BN0uFZ3PPDt+j9XE0x96s5DNKpfIg2zNsz7UUM1U6pkuQ6K53bS2tl5lqRIfM7W1RYc15rtss1PAyCzF+lyeKYHV9wuWsWMAzVLInN5WynX2hPI4LI4UXI9LfE0pLj95S8hScFNV5e6aih8dUwYN1ukoXbyDy6qn4vGUp2wMOLeM8s8LXkL5GcBRXgtNXZXjSOLxZ5ZQkdiDJWDSgGoWYlGN0VKlaoc0v60HMa/M/IxIDDAuzDsN5CPpN4zoHS1djV4245OYyskkn8DfJAIbJQqZGW7U78dAWFpBO9bXPWBTKWzwkTiIu7/KyCX81qeA9elB56gMktrY2HjCtr/gD/YoBbXd8M9Pzn48OaMXrR7ql5HAKfGx4G7jh1EVy7JeAb/RdGW04bT8QOAuI7GMdBeL9eMS3E98zS0HiStQwsbm/bbWOR2VBVeiqI+NXeGZMduKUjxPgcblRozbSoqJ12quXp7l9pXbOsZpakhh+xxeV8HnTn8RUdBM0JGezvYPpMw/l3OlT0IX/VlZRZmmAmuVzeTwANHiiRUegB1ar6KaA7IiR1ZI8mSRUy4jfn3DDgYtfhCqaR56+sumcxxqjWUrbxyloW1bEBjhf5gIXxAXGXHLFzHnwzkdHJynjadporQUbYHF+RvRinmZBjNfbzMtb/js+dNqYbTQ4Zz81QoE95qDP41aaaK2EoAY0scoYthammmmsTQ/9ehDnX5SMy6FzhfP288cz6ulha/fmvTxPE79WHbjC+SBaahq07uLCjV4cvpGhD7WSMFV6x+QaxdMTTBSSdWoUolLWDxp/jwSkM4G9ZukYX0o6lZ03kZsdbHaZ0N7p1XhsZagNa+Q0kFJQKipoLL49i/0+QGzbLfLYxhmzYoiV0c7O1MQ2XKM4N2aSXtSOxL8QA4GfycGjVDFDprBOOPhziOOdev5bWBUebvNk/DpAUUJmGhsHTWkqFlBlAi2AFfpskH27PCwA67Q3BVVwCF841QBMIPjBIstQaCGFs8trZgov7vds7ZV99aRV9OCpXP9hq5XZFci9UsZKz4R9gvn/1IB2jLPeFnMhjauyq3nuaCSh938BMQAgoyFh9HW15UvUg9tF8mf1NncvcEgGVt9rUHz0VEZ4HT7pH8HGLWidFWofVw5wH1r9OAUS0z2iCSoAXxrezvNtoFNWZmHnMIJaIJh14mFJqQloCS7e89o43aP7nGvHtYpCagiBEdy2J7hu9MTDLUhPpXt9vOL18BVHd/3FAPlDt8F9/CjtfybGbC6fYdtQ98/ZU8HvfPH+D74kLjl6Ql5KAdx9wgsJQTpBRFPCUxG2bLGhxHf/jTYfrF9+c1PHmxPsxs/IWZgtBDp0/VWzw0HtUvmLZHe8MTDg5q5vSCJBk4Jb70MU70Y713QlV1wCUhnXsE5YiM624/22Uf73FoniGTIQcxSzVH9fkhrzY1QGlm5NHxa5+UdejPHtpp/JwAQhoWRyrMUj04eIfrWbgYuee4tNEPhB4WviOF7gyKNeQCuNf1LBRTf6H/rQP87BlJ77pnntbMSBDN9Z2kxIs2hEoGNRoLFWLMcOhaK/ucb00YcaGI0I4TeBVs0hxvGLkBvDf3NxDf+F0FjZ0k="}""")
for _n, _d in _FILES.items():
    open(f"{APP}/ap3d/{_n}", "wb").write(zlib.decompress(base64.b64decode(_d)))
if APP not in sys.path:
    sys.path.insert(0, APP)
print(subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader",
                     shell=True, capture_output=True, text=True).stdout.strip() or "SIN GPU: cambia el entorno a GPU")
import psutil
print(f"RAM del sistema: {psutil.virtual_memory().total/2**30:.0f} GB · CPU: {os.cpu_count()}")
print("Proyecto listo en", APP)

## 2. Instalar dependencias y Blender 4.2 LTS

In [ ]:
# @title 2. Instalar dependencias, Hunyuan3D-2 y Blender 4.2 LTS
USAR_CACHE_EN_DRIVE = False  # @param {type:"boolean"}
# @markdown Si lo activas, las piezas compiladas se guardan en Drive y la próxima sesión se instalan en segundos.
import os, subprocess, sys, glob, shutil, time
t0 = time.time()

def sh(cmd, check=True):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-3000:], r.stderr[-3000:])
        if check:
            raise RuntimeError("Fallo: " + cmd)
    return r.stdout

# --- sistema (Blender sin ventana necesita estas librerias)
sh("apt-get -qq update && apt-get -qq install -y libxi6 libxkbcommon0 libsm6 libxrender1 "
   "libxxf86vm1 libxfixes3 libgl1 libegl1 libglu1-mesa libopengl0 > /dev/null", check=False)

# --- Python
sh(f"{sys.executable} -m pip install -q 'gradio>=5.9,<6' 'diffusers==0.32.2' 'transformers==4.49.0' 'accelerate>=1.2,<1.6' "
   "einops omegaconf trimesh pymeshlab pygltflib xatlas rembg onnxruntime-gpu "
   "opencv-python-headless ninja pybind11 psutil hf_transfer 'huggingface_hub>=0.25'")
print(f"Paquetes listos ({time.time()-t0:.0f}s)")

# --- Hunyuan3D-2
if not os.path.exists("/content/Hunyuan3D-2"):
    sh("git clone -q --depth 1 https://github.com/Tencent/Hunyuan3D-2 /content/Hunyuan3D-2")
sh(f"cd /content/Hunyuan3D-2 && {sys.executable} -m pip install -q -e . --no-deps")

import torch
cap = "%d.%d" % torch.cuda.get_device_capability(0) if torch.cuda.is_available() else "8.0"
os.environ["TORCH_CUDA_ARCH_LIST"] = cap            # compila solo para tu GPU: mas rapido
os.environ["MAX_JOBS"] = str(os.cpu_count())
tag = f"py{sys.version_info.major}{sys.version_info.minor}_torch{torch.__version__.split('+')[0]}_sm{cap}"
cache = None
if USAR_CACHE_EN_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    cache = f"/content/drive/MyDrive/ap3d_cache/{tag}"
    os.makedirs(cache, exist_ok=True)

def have(mod):
    return subprocess.run([sys.executable, "-c", f"import {mod}"], capture_output=True).returncode == 0

if not have("custom_rasterizer"):
    whl = glob.glob(f"{cache}/custom_rasterizer*.whl") if cache else []
    if whl:
        sh(f"{sys.executable} -m pip install -q {whl[0]}")
        print("custom_rasterizer desde la cache de Drive")
    else:
        print("Compilando custom_rasterizer (3-5 min)...")
        d = "/content/Hunyuan3D-2/hy3dgen/texgen/custom_rasterizer"
        sh(f"cd {d} && {sys.executable} -m pip wheel -q . --no-build-isolation --no-deps -w /content/whl")
        w = glob.glob("/content/whl/custom_rasterizer*.whl")[0]
        sh(f"{sys.executable} -m pip install -q {w}")
        if cache:
            shutil.copy(w, cache)

d = "/content/Hunyuan3D-2/hy3dgen/texgen/differentiable_renderer"
if not glob.glob(f"{d}/mesh_processor*.so"):
    so = glob.glob(f"{cache}/mesh_processor*.so") if cache else []
    if so:
        shutil.copy(so[0], d)
    else:
        sh(f"cd {d} && c++ -O3 -Wall -shared -std=c++11 -fPIC "
           f"$({sys.executable} -m pybind11 --includes) mesh_processor.cpp "
           f"-o mesh_processor$({sys.executable}-config --extension-suffix 2>/dev/null || "
           f"python3-config --extension-suffix)", check=False)
        if not glob.glob(f"{d}/mesh_processor*.so"):
            sh(f"cd {d} && {sys.executable} setup.py build_ext --inplace", check=False)
        if cache:
            for f in glob.glob(f"{d}/mesh_processor*.so"):
                shutil.copy(f, cache)
print(f"Hunyuan3D-2 listo ({time.time()-t0:.0f}s)")

# --- Blender 4.2 LTS
if not os.path.exists("/content/blender/blender"):
    url = "https://download.blender.org/release/Blender4.2/blender-4.2.3-linux-x64.tar.xz"
    sh(f"wget -q {url} -O /content/blender.tar.xz && tar -xf /content/blender.tar.xz -C /content "
       "&& mv /content/blender-4.2.3-linux-x64 /content/blender && rm /content/blender.tar.xz")
print(sh("/content/blender/blender -b --version | head -1").strip())
print(f"Instalación lista en {time.time()-t0:.0f}s")

## 3. Comprobar GPU, Blender y modelos de IA (carga todo en RAM y GPU)

In [ ]:
# @title 3. Comprobar GPU, descargar modelos, precargarlos en la RAM y cargarlos en la GPU
CARGAR_MULTIVISTA = True  # @param {type:"boolean"}
CARGAR_TEXTURA = True  # @param {type:"boolean"}
import os, sys, time
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
sys.path.insert(0, "/content/ap3d_app")
sys.path.insert(0, "/content/Hunyuan3D-2")
# Hunyuan3D-2 necesita transformers 4.49: con versiones nuevas su codificador
# de imagen (DINOv2) no carga ("Missing key(s) ... main_image_encoder").
import importlib.metadata as _md, subprocess
if _md.version("transformers") != "4.49.0" or not _md.version("diffusers").startswith("0.32"):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers==4.49.0",
                    "diffusers==0.32.2", "accelerate>=1.2,<1.6"], check=True)
    if "transformers" in sys.modules:
        raise SystemExit("Versiones corregidas. Ve a Entorno de ejecución › Reiniciar sesión "
                         "y vuelve a ejecutar la celda 1 y después esta celda 3.")
import torch
from huggingface_hub import snapshot_download
from ap3d import config
from ap3d.engine import get_engine

print("CUDA:", torch.cuda.is_available())
P = config.detect()
print(config.describe(P))

t = time.time()
pats = ["hunyuan3d-dit-v2-0/*"]
if CARGAR_TEXTURA:
    pats += ["hunyuan3d-paint-v2-0/*", "hunyuan3d-delight-v2-0/*"]
snapshot_download("tencent/Hunyuan3D-2", allow_patterns=pats, max_workers=16)
if CARGAR_MULTIVISTA:
    snapshot_download("tencent/Hunyuan3D-2mv", allow_patterns=["hunyuan3d-dit-v2-mv/*"], max_workers=16)
print(f"Modelos descargados ({time.time()-t:.0f}s)")

eng = get_engine()
if P["preload_weights"]:
    eng.preload_weights()          # deja todos los pesos en la RAM
eng.load(want_mv=CARGAR_MULTIVISTA, want_paint=CARGAR_TEXTURA)
print("Todo listo. Sigue con la celda 4 o 5.")

## 4. Estudio

In [ ]:
# @title 4. Estudio (dentro del notebook)
import sys; sys.path.insert(0, "/content/ap3d_app")
from ap3d.app import launch_inline
launch_inline(port=7860)

## 5. Abrir la página web con contraseña

In [ ]:
# @title 5. Abrir la página web con contraseña (para el celular)
CONTRASENA = ""  # @param {type:"string"}
# @markdown Déjala vacía para generar una al azar. El usuario es **ap**. La página deja de funcionar al cerrar la sesión de Colab.
import sys; sys.path.insert(0, "/content/ap3d_app")
from ap3d.app import launch_web
url, user, pw = launch_web(CONTRASENA, user="ap", port=7861)
print("Página web:", url)
print("Usuario:", user, "· Contraseña:", pw)

## 6. Guardar en Drive

In [ ]:
# @title 6. Guardar todos los resultados en Google Drive
CARPETA = "AP_Model_3D"  # @param {type:"string"}
import os, shutil, glob
from google.colab import drive
if not os.path.exists("/content/drive/MyDrive"):
    drive.mount("/content/drive")
dst = f"/content/drive/MyDrive/{CARPETA}"
os.makedirs(dst, exist_ok=True)
n = 0
for z in glob.glob("/content/ap3d_salidas/*.zip"):
    if not os.path.exists(os.path.join(dst, os.path.basename(z))):
        shutil.copy(z, dst); n += 1
print(f"{n} modelos nuevos copiados a {dst}")

## 7. Memoria

In [ ]:
# @title 7. (Opcional) Liberar GPU / ver uso de memoria
LIBERAR_MODELOS = False  # @param {type:"boolean"}
import gc, torch, psutil, sys
sys.path.insert(0, "/content/ap3d_app")
from ap3d.engine import get_engine
e = get_engine()
if LIBERAR_MODELOS:
    e.shape = e.shape_mv = e.paint = None; e.ready = False
    gc.collect(); torch.cuda.empty_cache()
e.log_mem()